In [ ]:
!pip install transformers==4.44.2 peft==0.11.1 --force-reinstall -q
!pip install -U -q accelerate datasets
!pip install -q packaging ninja timm einops bitsandbytes decord gdown scipy
!pip install -q rouge_score evaluate bert-score underthesea
!pip install --upgrade nltk -q

from pathlib import Path
WORKING = Path("/kaggle/working/")
import subprocess
if not (WORKING / 'InternVL').exists():
    subprocess.run(["git", "clone", "https://github.com/OpenGVLab/InternVL.git", str(WORKING / 'InternVL')])
!pip install -q -e {WORKING / 'InternVL/internvl_chat/'}


In [ ]:
import os, json, shutil
from pathlib import Path

WORKING = Path("/kaggle/working/")

MODEL_PATH = Path("/kaggle/input/models/bixunhong/intern/other/default/1/InternVL_2")

FINETUNE_CKPT = Path("/kaggle/input/models/maituananh511/internvl2-1b-finetune/pytorch/default/1/InternVL2_1B_train")

VI_CHART_DATASET_PATH  = Path("/kaggle/input/datasets/maituananh511/dataset-chart-vqa/vi_chart_dataset")
VIETNAMESE_DATA_PATH   = Path("/kaggle/input/datasets/maituananh511/data-vietnamese/Data Vietnamese")
VIETNAMESE_IMAGES_PATH = VIETNAMESE_DATA_PATH / "images"
VIETNAMESE_JSONL_PATH  = VIETNAMESE_DATA_PATH / "viet_chart_vqa.jsonl"

CHART_TEST_N = 500
VIETNAMESE_N = 200
EVAL_TOTAL   = CHART_TEST_N + VIETNAMESE_N

print("MODEL_PATH     exists:", MODEL_PATH.exists())
print("FINETUNE_CKPT  exists:", FINETUNE_CKPT.exists())
print("VI_CHART       exists:", VI_CHART_DATASET_PATH.exists())
print("VIETNAMESE     exists:", VIETNAMESE_DATA_PATH.exists())

In [ ]:
from datasets import load_from_disk
from PIL import Image
import json as _json

def normalize_turn(turn):
    if isinstance(turn, str):
        return {'role': 'assistant', 'content': turn}
    if isinstance(turn, dict):
        role = turn.get('role') or turn.get('from', '')
        if role in ('human', 'user'):      role = 'user'
        elif role in ('gpt', 'assistant'): role = 'assistant'
        for rk in ('assistant', 'user', 'human', 'gpt'):
            if rk in turn and 'content' not in turn and 'role' not in turn:
                role = 'assistant' if rk in ('assistant', 'gpt') else 'user'
                return {'role': role, 'content': str(turn[rk])}
        return {'role': role, 'content': str(turn.get('content') or turn.get('value', ''))}
    return {'role': 'assistant', 'content': str(turn)}

vi_chart_dataset = load_from_disk(str(VI_CHART_DATASET_PATH))
print("vi_chart_dataset:", vi_chart_dataset)
chart_test_raw   = vi_chart_dataset['test']
chart_n          = min(CHART_TEST_N, len(chart_test_raw))
chart_test_items = [chart_test_raw[i] for i in range(chart_n)]
print(f"vi_chart test   : {chart_n} samples")

vietnamese_records = []
with open(VIETNAMESE_JSONL_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            vietnamese_records.append(_json.loads(line))
print(f"Vietnamese records: {len(vietnamese_records)} loaded")

vn_test_items = []
for record in reversed(vietnamese_records):
    if len(vn_test_items) >= VIETNAMESE_N:
        break
    img_path = VIETNAMESE_IMAGES_PATH / record['image']
    try:
        image = Image.open(img_path).convert('RGB')
    except Exception as e:
        print(f'Warning: {img_path}: {e}')
        continue
    convs = [normalize_turn(t) for t in record['conversations']]
    pairs = [(convs[i], convs[i+1]) for i in range(0, len(convs)-1, 2)]
    for idx, (q, a) in enumerate(pairs):
        if len(vn_test_items) >= VIETNAMESE_N:
            break
        rid = record['id'] if len(pairs) == 1 else f"{record['id']}_q{idx}"
        vn_test_items.append({'id': rid, 'image': image, 'conversations': [q, a]})
print(f"vietnamese test  : {len(vn_test_items)} samples")

eval_dataset = [x for x in (chart_test_items + vn_test_items) if x.get('image') is not None]


In [ ]:
import sys, torch
sys.path.insert(0, str(WORKING / 'InternVL/internvl_chat'))
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoTokenizer
from internvl.model.internvl_chat import InternVLChatConfig, InternVLChatModel

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def build_transform(input_size):
    return T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float('inf')
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_ar  = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_ar)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=12, image_size=448, use_thumbnail=False):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height
    target_ratios = set(
        (i, j) for n in range(min_num, max_num + 1)
        for i in range(1, n + 1) for j in range(1, n + 1)
        if min_num <= i * j <= max_num
    )
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])
    target_ar = find_closest_aspect_ratio(aspect_ratio, target_ratios, orig_width, orig_height, image_size)
    target_w = image_size * target_ar[0]
    target_h = image_size * target_ar[1]
    blocks   = target_ar[0] * target_ar[1]
    resized  = image.resize((target_w, target_h))
    processed = []
    for i in range(blocks):
        box = (
            (i % (target_w  // image_size)) * image_size,
            (i // (target_w // image_size)) * image_size,
            ((i % (target_w  // image_size)) + 1) * image_size,
            ((i // (target_w // image_size)) + 1) * image_size,
        )
        processed.append(resized.crop(box))
    if use_thumbnail and len(processed) != 1:
        processed.append(image.resize((image_size, image_size)))
    return processed

def load_image(image_file, input_size=448, max_num=12):
    image = Image.open(image_file).convert('RGB') if isinstance(image_file, str) else image_file
    transform    = build_transform(input_size=input_size)
    images       = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    pixel_values = torch.stack([transform(img) for img in images])
    return pixel_values

def load_internvl(model_path, tokenizer_path=None):
    """
    Load InternVL2 model.
    - model_path    : thư mục chứa weights (MODEL_PATH hoặc FINETUNE_CKPT)
    - tokenizer_path: nếu None thì dùng MODEL_PATH (base tokenizer)
    """
    cfg = InternVLChatConfig.from_pretrained(str(model_path))
    if isinstance(cfg, tuple): cfg = cfg[0]
    cfg._attn_implementation = 'eager'
    if hasattr(cfg, 'llm_config'):
        cfg.llm_config._attn_implementation = 'eager'

    tok_path = tokenizer_path if tokenizer_path else MODEL_PATH
    tok = AutoTokenizer.from_pretrained(str(tok_path), trust_remote_code=True, use_fast=False)
    tok.pad_token    = tok.eos_token
    tok.padding_side = 'left'

    mdl = InternVLChatModel.from_pretrained(
        str(model_path), config=cfg,
        torch_dtype=torch.float16, low_cpu_mem_usage=False,
        trust_remote_code=True, attn_implementation='eager'
    ).eval().cuda()
    mdl.config.pad_token_id = tok.eos_token_id
    return mdl, tok

GEN_CFG_PRE = dict(max_new_tokens=256, do_sample=False, num_beams=3, repetition_penalty=2.0)
GEN_CFG_FT  = dict(max_new_tokens=256, do_sample=False, num_beams=1, repetition_penalty=2.5,
                   no_repeat_ngram_size=5)


In [ ]:
import json
from pathlib import Path

cfg = json.load(open(FINETUNE_CKPT / 'config.json'))
print(json.dumps(cfg, indent=2))

In [ ]:
import torch
from safetensors import safe_open
from safetensors.torch import save_file
from pathlib import Path
import shutil

FIXED_CKPT = WORKING / "InternVL2_1B_fixed"
FIXED_CKPT.mkdir(parents=True, exist_ok=True)

print("Loading safetensors...")
tensors = {}
with safe_open(str(FINETUNE_CKPT / 'model.safetensors'), framework='pt', device='cpu') as f:
    for k in f.keys():
        new_key = k.replace('language_model.base_model.model.', 'language_model.')
        tensors[new_key] = f.get_tensor(k)
        if k != new_key:
            print(f"  Renamed: {k[:80]}")

print(f"\nTổng keys: {len(tensors)}")

emb_key = 'language_model.model.embed_tokens.weight'
if emb_key in tensors:
    print(f"embed_tokens max: {tensors[emb_key].float().abs().max().item():.6f}")

print("\nSaving fixed safetensors...")
save_file(tensors, str(FIXED_CKPT / 'model.safetensors'), metadata={"format": "pt"})
print(" Saved với metadata format=pt")

for fname in FINETUNE_CKPT.iterdir():
    if fname.suffix != '.safetensors' and fname.name not in ['optimizer.pt']:
        shutil.copy2(str(fname), str(FIXED_CKPT / fname.name))
        print(f" Copied {fname.name}")

for fname in MODEL_PATH.glob('token*'):
    shutil.copy2(str(fname), str(FIXED_CKPT))
for fname in MODEL_PATH.glob('special_tokens*'):
    shutil.copy2(str(fname), str(FIXED_CKPT))
for fname in MODEL_PATH.glob('*.py'):
    shutil.copy2(str(fname), str(FIXED_CKPT))


In [ ]:
from safetensors import safe_open
import torch, sys
sys.path.insert(0, str(WORKING / 'InternVL/internvl_chat'))
from internvl.model.internvl_chat import InternVLChatConfig, InternVLChatModel

cfg = InternVLChatConfig.from_pretrained(str(FIXED_CKPT))
if isinstance(cfg, tuple): cfg = cfg[0]
empty_mdl = InternVLChatModel(cfg)
expected_keys = set(empty_mdl.state_dict().keys())

actual_keys = set()
with safe_open(str(FIXED_CKPT / 'model.safetensors'), framework='pt', device='cpu') as f:
    actual_keys = set(f.keys())

missing  = expected_keys - actual_keys
extra    = actual_keys - expected_keys

print(f"Expected : {len(expected_keys)}")
print(f"Actual   : {len(actual_keys)}")
print(f"Missing  : {len(missing)}")
print(f"Extra    : {len(extra)}")

if missing:
    print("\nMissing keys (sample 10):")
    for k in list(missing)[:10]:
        print(f"  {k}")
if extra:
    print("\nExtra keys (sample 10):")
    for k in list(extra)[:10]:
        print(f"  {k}")

del empty_mdl

In [ ]:
import torch, shutil, sys
from safetensors import safe_open
from safetensors.torch import save_file
sys.path.insert(0, str(WORKING / 'InternVL/internvl_chat'))
from internvl.model.internvl_chat import InternVLChatConfig, InternVLChatModel
from transformers import AutoTokenizer

FIXED_CKPT = WORKING / "InternVL2_1B_fixed"
if FIXED_CKPT.exists():
    shutil.rmtree(str(FIXED_CKPT))
FIXED_CKPT.mkdir(parents=True, exist_ok=True)

LORA_RANK  = 16
LORA_ALPHA = 32
SCALING    = LORA_ALPHA / LORA_RANK  # 2.0

print("Loading base model...")
cfg = InternVLChatConfig.from_pretrained(str(FINETUNE_CKPT))  
if isinstance(cfg, tuple): cfg = cfg[0]
cfg._attn_implementation = 'eager'
if hasattr(cfg, 'llm_config'):
    cfg.llm_config._attn_implementation = 'eager'

model = InternVLChatModel.from_pretrained(
    str(MODEL_PATH),        
    config=cfg,             
    torch_dtype=torch.float16,
    trust_remote_code=True,
    attn_implementation='eager',
    ignore_mismatched_sizes=True,  
)
print(" Base model loaded")
print("Base embed max:", model.language_model.model.embed_tokens.weight.float().abs().max().item())

print("\nLoading finetuned tensors...")
raw = {}
with safe_open(str(FINETUNE_CKPT / 'model.safetensors'), framework='pt', device='cpu') as f:
    for k in f.keys():
        raw[k] = f.get_tensor(k)
print(f"Raw keys: {len(raw)}")

model_keys = set(model.state_dict().keys())
print(f"Model keys: {len(model_keys)}")

state = model.state_dict()
merged_count = 0
lora_A_keys = [k for k in raw if 'lora_A.default.weight' in k]

for ka in lora_A_keys:
    prefix = ka.replace('language_model.base_model.model.', 'language_model.')
    prefix = prefix.replace('.lora_A.default.weight', '')
    
    kb      = ka.replace('lora_A', 'lora_B')
    kb_base = ka.replace('lora_A.default.weight', 'base_layer.weight')
    
    if kb not in raw or kb_base not in raw:
        continue
    
    target_key = prefix + '.weight'
    if target_key not in state:
        print(f"  SKIP (not in model): {target_key}")
        continue
    
    A     = raw[ka].float()
    B     = raw[kb].float()
    base  = raw[kb_base].float()
    delta = (B @ A) * SCALING
    state[target_key] = (base + delta).half()
    merged_count += 1

print(f" Merged {merged_count} LoRA layers")

bias_keys = [k for k in raw if 'base_layer.bias' in k]
for k in bias_keys:
    target = k.replace('language_model.base_model.model.', 'language_model.')
    target = target.replace('.base_layer.bias', '.bias')
    if target in state:
        state[target] = raw[k]

for k, v in raw.items():
    if 'lora_A' in k or 'lora_B' in k or 'base_layer' in k or 'base_model' in k:
        continue
    if k in state:
        state[k] = v

missing, unexpected = model.load_state_dict(state, strict=False)
print(f"Missing keys   : {len(missing)}")
print(f"Unexpected keys: {len(unexpected)}")

print("Embed max sau merge:", model.language_model.model.embed_tokens.weight.float().abs().max().item())

print("\nSaving...")
model.save_pretrained(str(FIXED_CKPT))
for fname in MODEL_PATH.glob('token*'):
    shutil.copy2(str(fname), str(FIXED_CKPT))
for fname in MODEL_PATH.glob('special_tokens*'):
    shutil.copy2(str(fname), str(FIXED_CKPT))
for fname in MODEL_PATH.glob('*.py'):
    shutil.copy2(str(fname), str(FIXED_CKPT))

print(f" Done: {FIXED_CKPT}")
del model
torch.cuda.empty_cache()

In [ ]:
ft_mdl, ft_tok = load_internvl(FIXED_CKPT, tokenizer_path=MODEL_PATH)

In [ ]:
import gc, matplotlib.pyplot as plt
from PIL import Image

NUM_SAMPLES  = 3
sample_items = [x for x in eval_dataset if x.get('image') is not None][:NUM_SAMPLES]

for i, item in enumerate(sample_items):
    plt.figure(figsize=(5, 4))
    plt.imshow(item['image'])
    plt.axis('off')
    plt.title(f"Sample {i} — {item['id']}")
    plt.show()
    print(f"Q : {item['conversations'][0].get('content', '')}")
    print(f"GT: {item['conversations'][1].get('content', '')}")
    print('='*60)

print("\n" + "="*60)
print("EVAL PRETRAIN (MODEL_PATH)")
print("="*60)
pre_mdl, pre_tok = load_internvl(MODEL_PATH)
pretrain_responses = []
for item in sample_items:
    pv = load_image(item['image'], max_num=12).to(torch.float16).cuda()
    q  = str(item['conversations'][0].get('content') or item['conversations'][0].get('value', ''))
    r  = pre_mdl.chat(pre_tok, pv, f'<image>\n{q}', GEN_CFG_PRE)
    pretrain_responses.append(r)
pre_mdl = None
gc.collect()
print(" Pretrain xong")

print("\n" + "="*60)
print("EVAL FINETUNED (FINETUNE_CKPT)")
print("="*60)
ft_mdl, ft_tok = load_internvl(FIXED_CKPT, tokenizer_path=MODEL_PATH)
finetuned_responses = []
for item in sample_items:
    pv = load_image(item['image'], max_num=12).to(torch.float16).cuda()
    q  = str(item['conversations'][0].get('content') or item['conversations'][0].get('value', ''))
    r  = ft_mdl.chat(ft_tok, pv, f'<image>\n{q}', GEN_CFG_FT)
    finetuned_responses.append(r)
ft_mdl = None
gc.collect()
print(" Finetuned xong")

print("\n" + "="*60)
print("SO SÁNH PRETRAIN vs FINETUNED")
print("="*60)
for i, item in enumerate(sample_items):
    q  = item['conversations'][0].get('content', '')
    gt = item['conversations'][1].get('content', '')
    print(f"\n[Sample {i}] {item['id']}")
    print(f"   Q        : {q}")
    print(f"   GT       : {gt}")
    print(f"   Pretrain : {pretrain_responses[i]}")
    print(f"   Finetuned: {finetuned_responses[i]}")

In [ ]:
from safetensors import safe_open
import torch

with safe_open(str(FINETUNE_CKPT / 'model.safetensors'), framework='pt', device='cpu') as f:
    keys = list(f.keys())
    print(f"Tổng keys: {len(keys)}")
    
    embed_keys = [k for k in keys if 'embed_tokens' in k]
    lm_keys    = [k for k in keys if 'lm_head' in k]
    print(f"embed_tokens keys: {embed_keys}")
    print(f"lm_head keys     : {lm_keys}")
    
    for k in embed_keys + lm_keys:
        t = f.get_tensor(k)
        print(f"  {k}: shape={t.shape}, max={t.float().abs().max().item():.6f}, mean={t.float().abs().mean().item():.6f}")
    
    llm_keys = [k for k in keys if 'language_model' in k and 'weight' in k][:3]
    for k in llm_keys:
        t = f.get_tensor(k)
        print(f"  {k}: max={t.float().abs().max().item():.6f}")

In [ ]:
import os, nltk, gc
import pandas as pd
import numpy as np
from tqdm import tqdm

nltk_path = '/usr/share/nltk_data'
os.makedirs(nltk_path, exist_ok=True)
nltk.data.path.append(nltk_path)
for pkg in ['punkt', 'wordnet', 'omw-1.4']:
    nltk.download(pkg, download_dir=nltk_path, quiet=True)

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score as nltk_meteor
from rouge_score import rouge_scorer
from underthesea import word_tokenize

METRICS = ['bleu', 'meteor', 'rouge1', 'rouge2', 'rougeL', 'bertscore']

def compute_bertscore(responses, references):
    try:
        import bert_score as bs_lib
        _, _, F1 = bs_lib.score(responses, references, lang='vi', verbose=False,
                                rescale_with_baseline=False)
        return F1.tolist()
    except Exception as e:
        print(f'BERTScore failed: {e}')
        return [0.0] * len(responses)

def evaluate_internvl(model, tokenizer, items, gen_cfg, model_name='model'):
    scorer_rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    smoothie     = SmoothingFunction().method1
    results      = []
    all_resp, all_ref = [], []

    for item in tqdm(items, desc=f'[{model_name}]'):
        image = item.get('image')
        if image is None:
            continue
        convs    = item['conversations']
        question = str(convs[0].get('content') or convs[0].get('value', ''))
        gt       = str(convs[1].get('content') or convs[1].get('value', ''))

        try:
            pv       = load_image(image, max_num=12).to(torch.float16).cuda()
            response = model.chat(tokenizer, pv, f'<image>\n{question}', gen_cfg)
        except Exception as e:
            print(f"  Lỗi {item['id']}: {e}")
            response = ''

        ref  = word_tokenize(gt,       format='text').split()
        hyp  = word_tokenize(response, format='text').split() if response else ['']

        try:
            bleu = sentence_bleu([ref], hyp, weights=(0.25,)*4, smoothing_function=smoothie)
        except ZeroDivisionError:
            bleu = 0.0
        try:
            meteor = float(nltk_meteor([ref], hyp))
        except Exception:
            meteor = 0.0

        r = scorer_rouge.score(gt, response)
        all_resp.append(response)
        all_ref.append(gt)
        results.append({
            'id':           item['id'],
            'question':     question,
            'ground_truth': gt,
            'response':     response,
            'bleu':         bleu,
            'meteor':       meteor,
            'rouge1':       r['rouge1'].fmeasure,
            'rouge2':       r['rouge2'].fmeasure,
            'rougeL':       r['rougeL'].fmeasure,
            'bertscore':    0.0,
        })

    df = pd.DataFrame(results)
    print(f'\nComputing BERTScore ({model_name}) ...')
    bert_f1 = compute_bertscore(all_resp, all_ref)
    df['bertscore'] = bert_f1

    avg = {m: df[m].mean() for m in METRICS}
    print(f'\n[{model_name}] Average scores ({len(df)} samples):')
    for k, v in avg.items():
        print(f'  {k:12s}: {v:.4f}')
    return df, avg

print(" evaluate_internvl defined")

In [ ]:
print("="*60)
print(f"EVAL FINETUNED [{len(eval_dataset)} samples]")
print("="*60)
ft_mdl, ft_tok = load_internvl(FIXED_CKPT, tokenizer_path=MODEL_PATH)
df_finetuned, avg_finetuned = evaluate_internvl(
    ft_mdl, ft_tok, eval_dataset, GEN_CFG_FT, 'Finetuned'
)
ft_mdl = None
gc.collect()
print(" Finetuned eval xong")

df_finetuned['model'] = 'finetuned'
csv_ft = WORKING / 'eval_finetuned.csv'
df_finetuned.to_csv(str(csv_ft), index=False, encoding='utf-8')
print(f" Saved: {csv_ft}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

METRICS     = ['bleu', 'meteor', 'rouge1', 'rouge2', 'rougeL', 'bertscore']
VINTERN_CSV = Path('/kaggle/input/datasets/maituananh511/700-vinternlora-result/debug_vintern_lora.csv')

print("Loading CSVs...")
df_finetuned = pd.read_csv(str(WORKING / 'eval_finetuned.csv'))
df_vintern   = pd.read_csv(str(VINTERN_CSV))
df_vintern.columns = [c.strip() for c in df_vintern.columns]
col_map = {c.lower(): c for c in df_vintern.columns}

df_v = df_vintern.rename(columns={col_map.get(m, m): m for m in METRICS if col_map.get(m)})

print(f"InternVL-FT : {len(df_finetuned)} samples")
print(f"Vintern-LoRA: {len(df_v)} samples")

avg_ft = {m: df_finetuned[m].mean() if m in df_finetuned.columns else 0.0 for m in METRICS}
avg_vt = {m: df_v[m].mean()         if m in df_v.columns         else 0.0 for m in METRICS}

all_models = {
    'InternVL-FT':  avg_ft,
    'Vintern-LoRA': avg_vt,
}

rows = []
for mname, avg in all_models.items():
    row = {'Model': mname}
    for m in METRICS:
        row[m.upper()] = round(avg.get(m, 0.0), 4)
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index('Model')

delta_row = {'Model': 'Delta (FT - VT)'}
for m in METRICS:
    delta_row[m.upper()] = round(avg_ft[m] - avg_vt[m], 4)
delta_df = pd.DataFrame([delta_row]).set_index('Model')

def highlight_best(s):
    return ['background-color: #d4edda; font-weight: bold' if v == s.max() else '' for v in s]
def highlight_worst(s):
    return ['background-color: #f8d7da' if v == s.min() else '' for v in s]
def highlight_delta(s):
    return ['color: green; font-weight: bold' if v > 0
            else ('color: red' if v < 0 else '') for v in s]

print('\n===== KẾT QUẢ SO SÁNH =====\n')
display(summary_df.style.apply(highlight_best).apply(highlight_worst))
print()
def highlight_delta_cell(v):
    if isinstance(v, float):
        return 'color: green; font-weight: bold' if v > 0 else ('color: red' if v < 0 else '')
    return ''

display(delta_df.style.map(highlight_delta_cell))
x      = np.arange(len(METRICS))
width  = 0.35
colors = ['#e07b54', '#2ecc71']

fig, ax = plt.subplots(figsize=(13, 5))
for idx, (mname, avg) in enumerate(all_models.items()):
    offset = (idx - 0.5) * width
    vals   = [avg.get(m, 0.0) for m in METRICS]
    bars   = ax.bar(x + offset, vals, width, label=mname, color=colors[idx])
    ax.bar_label(bars, fmt='%.3f', padding=2, fontsize=8, rotation=90)

ax.set_xticks(x)
ax.set_xticklabels([m.upper() for m in METRICS], fontsize=10)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.2)
ax.set_title('InternVL-FT vs Vintern-LoRA', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(str(WORKING / 'bar_ft_vs_vintern.png'), dpi=150)
plt.show()
print("Saved: bar_ft_vs_vintern.png")

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, m in enumerate(METRICS):
    ax = axes[i]
    if m in df_finetuned.columns:
        ax.hist(df_finetuned[m], bins=30, alpha=0.6, color='#e07b54', label='InternVL-FT')
        ax.axvline(df_finetuned[m].mean(), color='#e07b54', linestyle='--', linewidth=1.5,
                   label=f'FT  mean={df_finetuned[m].mean():.3f}')
    if m in df_v.columns:
        ax.hist(df_v[m], bins=30, alpha=0.6, color='#2ecc71', label='Vintern-LoRA')
        ax.axvline(df_v[m].mean(), color='#2ecc71', linestyle='--', linewidth=1.5,
                   label=f'VT  mean={df_v[m].mean():.3f}')
    ax.set_title(m.upper(), fontsize=11)
    ax.set_xlabel('Score')
    ax.set_ylabel('Count')
    ax.legend(fontsize=7)

plt.suptitle('Phân phối score: InternVL-FT vs Vintern-LoRA', fontsize=13)
plt.tight_layout()
plt.savefig(str(WORKING / 'histogram_ft_vs_vintern.png'), dpi=150)
plt.show()
print("Saved: histogram_ft_vs_vintern.png")

if 'id' in df_finetuned.columns and 'id' in df_v.columns:
    df_delta = df_finetuned[['id', 'question', 'ground_truth', 'response'] + METRICS].merge(
        df_v[['id'] + [m for m in METRICS if m in df_v.columns]],
        on='id', suffixes=('_ft', '_vt'), how='inner'
    )
    for m in METRICS:
        cf, cv = f'{m}_ft', f'{m}_vt'
        if cf in df_delta.columns and cv in df_delta.columns:
            df_delta[f'delta_{m}'] = df_delta[cf] - df_delta[cv]
    print(f"\nPer-sample merged: {len(df_delta)} samples")
    print("\nTop 5 InternVL-FT vượt trội (BERTScore):")
    display(df_delta.nlargest(5, 'delta_bertscore')[
        ['id', 'question', 'delta_bleu', 'delta_meteor', 'delta_bertscore']])
    print("\nTop 5 InternVL-FT kém hơn (BERTScore):")
    display(df_delta.nsmallest(5, 'delta_bertscore')[
        ['id', 'question', 'delta_bleu', 'delta_meteor', 'delta_bertscore']])

In [ ]:
from datetime import datetime

timestamp  = datetime.now().strftime('%Y%m%d_%H%M%S')
EXPORT_DIR = WORKING / f'export_InternVL2_eval_{timestamp}'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for fname, df in [
    ('eval_finetuned.csv',              df_finetuned),
    ('eval_summary_ft_vs_vintern.csv',  summary_df.reset_index()),
]:
    p = WORKING / fname
    df.to_csv(str(p), index=False, encoding='utf-8')
    shutil.copy(str(p), str(EXPORT_DIR / fname))
    print(f" {fname}")

if 'df_delta' in dir():
    p = WORKING / 'eval_comparison_ft_vs_vintern.csv'
    df_delta.to_csv(str(p), index=False, encoding='utf-8')
    shutil.copy(str(p), str(EXPORT_DIR / 'eval_comparison_ft_vs_vintern.csv'))
    print(" eval_comparison_ft_vs_vintern.csv")

for fname in ['bar_ft_vs_vintern.png', 'histogram_ft_vs_vintern.png']:
    src = WORKING / fname
    if src.exists():
        shutil.copy(str(src), str(EXPORT_DIR / fname))
        print(f" {fname}")

print(f"\n Export: {EXPORT_DIR}")
for f in sorted(EXPORT_DIR.rglob('*')):
    if f.is_file():
        print(f"  {f.relative_to(EXPORT_DIR)} ({f.stat().st_size/1e6:.2f} MB)")